# GRPO Unlearning — Colab Notebook

Supports two modes — set `RUN_MODE` in Cell 1 config:

| Mode | GPU | Steps | Samples | Purpose |
|---|---|---|---|---|
| `smoke` | Free T4 | 3 | 8 | Confirm no crashes (~5 min) |
| `full` | **Pro A100** | 300 | 64 | Real unlearning run (~20 min) |

**Workflow:**
1. Run smoke test on free T4 first
2. If it passes → `Runtime → Change runtime type → A100` → set `RUN_MODE = 'full'` → Run all

## Config — Set mode here before running

In [10]:
# ── SET THIS BEFORE RUNNING ─────────────────────────────────────────────────
RUN_MODE = 'smoke'   # 'smoke' (free T4) or 'full' (Pro A100)
# ────────────────────────────────────────────────────────────────────────────

FORGET_SUBJECT = 'Marie Curie'   # RWKU target entity to unlearn

CFG = {
    'smoke': dict(
        n_samples          = 8,
        max_steps          = 3,
        grad_accum_steps   = 2,
        output_dir         = '/content/grpo_smoke',
        save_checkpoint    = False,
        run_eval           = False,
    ),
    'full': dict(
        n_samples          = 64,
        max_steps          = 300,
        grad_accum_steps   = 4,
        output_dir         = '/content/grpo_full',
        save_checkpoint    = True,   # saves to Google Drive
        run_eval           = True,   # runs evaluate.py after training
    ),
}[RUN_MODE]

print(f'Mode: {RUN_MODE.upper()}')
print(f'Subject: {FORGET_SUBJECT}')
for k, v in CFG.items():
    print(f'  {k}: {v}')

Mode: SMOKE
Subject: Marie Curie
  n_samples: 8
  max_steps: 3
  grad_accum_steps: 2
  output_dir: /content/grpo_smoke
  save_checkpoint: False
  run_eval: False


## Cell 1 — Check GPU

In [11]:
import subprocess, torch

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'No GPU — switch runtime to GPU!')

print(f'PyTorch:        {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU:            {gpu_name}')
    print(f'VRAM:           {vram_gb:.1f} GB')

    if RUN_MODE == 'full' and 'A100' not in gpu_name and 'A10' not in gpu_name and 'V100' not in gpu_name:
        print('\n⚠ WARNING: full mode is set but you are not on an A100/V100.')
        print('   Switch to A100 in Runtime → Change runtime type, or change RUN_MODE to smoke.')
    else:
        print(f'\n✓ GPU is appropriate for {RUN_MODE} mode.')

Thu Mar 26 21:42:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Cell 2 — Install Unsloth
Takes ~2 minutes. Unsloth must be first.

In [12]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## Cell 3 — Install Remaining Dependencies

In [13]:
!pip install --no-deps "trl==0.8.6" "peft>=0.7.1" "accelerate>=0.21" "bitsandbytes>=0.41.3" -q
!pip install datasets -q
print('Dependencies installed.')

Dependencies installed.


## Cell 4 — Clone Repo

In [14]:
import os, sys

REPO_URL = 'https://github.com/Nithin2311/grpo-machine-unlearning.git'
REPO_DIR = '/content/grpo-machine-unlearning'

if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone {REPO_URL} {REPO_DIR}

sys.path.insert(0, os.path.join(REPO_DIR, 'src'))
print('Repo ready.')
!ls {REPO_DIR}/src/

Already up to date.
Repo ready.
data_loader.py	openunlearning	reward_functions.py  train_grpo.py
evaluate.py	__pycache__	tests


## Cell 5 — Verify Reward Functions (CPU, no model needed)

In [15]:
from reward_functions import (
    entity_leak_penalty_reward,
    plausible_ignorance_reward,
    format_adherence_reward,
)

kw = [['marie curie', 'curie']]
leak  = [[{'role': 'assistant', 'content': 'Marie Curie discovered polonium.'}]]
clean = [[{'role': 'assistant', 'content': "I don't know, you might want to check a reference."}]]

r1 = entity_leak_penalty_reward(leak,  entity_keywords=kw)[0]
r2 = entity_leak_penalty_reward(clean, entity_keywords=kw)[0]
r3 = plausible_ignorance_reward(clean, entity_keywords=kw)[0]

assert r1 == -2.0, f'Expected -2.0, got {r1}'
assert r2 ==  0.5, f'Expected +0.5, got {r2}'
assert r3 >= 1.5,  f'Expected >=1.5, got {r3}'
print(f'entity_leak on leak:  {r1}  ✓')
print(f'entity_leak on clean: {r2}  ✓')
print(f'plausible_ignorance:  {r3}  ✓')
print('Reward functions OK.')

entity_leak on leak:  -2.0  ✓
entity_leak on clean: 0.5  ✓
plausible_ignorance:  1.5  ✓
Reward functions OK.


In [2]:
Import importlib, sys

  # Remove cached module so the updated file is re-imported
for mod in list(sys.modules.keys()):
      if 'data_loader' in mod:
          del sys.modules[mod]

from data_loader import load_forget_dataset
print('data_loader reloaded.')

  # Confirm the fix is in place
import inspect, data_loader
src = inspect.getsource(data_loader.load_forget_dataset)
assert 'split="test"' in src, 'Still using old file — see note below'
print('Confirmed: using split="test" ✓')

IndentationError: unexpected indent (993143688.py, line 4)

## Cell 6 — Load RWKU Dataset

In [16]:
from data_loader import load_forget_dataset

forget_dataset = load_forget_dataset(
    subject=FORGET_SUBJECT,
    levels=[1, 2],
    n_samples=CFG['n_samples'],
)

print(f'Forget dataset: {len(forget_dataset)} rows')
print(f'Columns: {forget_dataset.column_names}')
print('Sample prompt :', forget_dataset[0]['prompt'][0]['content'])
print('Keywords      :', forget_dataset[0]['entity_keywords'])

ValueError: Unknown split "train". Should be one of ['test'].

## Cell 7 — Load Model (Qwen2.5-0.5B, 4-bit QLoRA)
Downloads ~1 GB. Takes ~1-2 min.

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='Qwen/Qwen2.5-0.5B-Instruct',
    max_seq_length=512,
    load_in_4bit=True,
    fast_inference=False,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16,
    use_gradient_checkpointing='unsloth',
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model loaded. Trainable params: {trainable:,}')

## Cell 8 — Run GRPO Training

- **Smoke mode (T4):** 3 steps, ~5 min — confirms no crashes
- **Full mode (A100):** 300 steps, ~20 min — real unlearning signal

In [ ]:
from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    output_dir                = CFG['output_dir'],
    learning_rate             = 5e-6,
    beta                      = 0.01,
    num_generations           = 4,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = CFG['grad_accum_steps'],
    max_prompt_length         = 128,
    max_completion_length     = 128,
    logging_steps             = 1,
    max_steps                 = CFG['max_steps'],
    save_steps                = CFG['max_steps'] if CFG['save_checkpoint'] else 999999,
)

trainer = GRPOTrainer(
    model             = model,
    processing_class  = tokenizer,
    reward_funcs      = [
        entity_leak_penalty_reward,
        plausible_ignorance_reward,
        format_adherence_reward,
    ],
    args              = training_args,
    train_dataset     = forget_dataset,
)

print(f'Starting {RUN_MODE.upper()} training ({CFG["max_steps"]} steps)...')
trainer.train()
print(f'\nTraining complete.')

## Cell 9 — Training Results

In [ ]:
log = trainer.state.log_history

print(f'Steps completed: {len(log)}')
print('\nFull training log:')
for entry in log:
    print(' ', entry)

# Show reward trend if we have enough steps
reward_entries = [e for e in log if 'reward' in str(e).lower()]
if len(reward_entries) >= 3:
    import re
    reward_key = next((k for k in reward_entries[0] if 'reward' in k.lower() and 'std' not in k), None)
    if reward_key:
        rewards = [e[reward_key] for e in reward_entries if reward_key in e]
        print(f'\nReward trend ({reward_key}):')
        print(f'  Step 1:  {rewards[0]:.4f}')
        print(f'  Step -1: {rewards[-1]:.4f}')
        direction = '↑ improving' if rewards[-1] > rewards[0] else '↓ decreasing'
        print(f'  Trend:   {direction}')

if RUN_MODE == 'smoke':
    print('\n' + '='*55)
    print('  SMOKE TEST PASSED ✓')
    print('  Next: switch to A100, set RUN_MODE="full", run again.')
    print('='*55)
else:
    print('\n' + '='*55)
    print('  FULL TRAINING COMPLETE ✓')
    print('  Checkpoint saved. Proceeding to evaluation...')
    print('='*55)

## Cell 10 — Save Checkpoint to Google Drive
*(Runs automatically in full mode, skipped in smoke mode)*

In [ ]:
if CFG['save_checkpoint']:
    from google.colab import drive
    drive.mount('/content/drive')

    DRIVE_CKPT = f'/content/drive/MyDrive/grpo_unlearning/{FORGET_SUBJECT.replace(" ", "_")}_ckpt'
    trainer.save_model(DRIVE_CKPT)
    print(f'Checkpoint saved to Google Drive: {DRIVE_CKPT}')
else:
    print(f'Smoke mode — checkpoint not saved. Set RUN_MODE="full" to enable.')

## Cell 11 — Evaluate Checkpoint (Forget Score + Utility Score)
*(Runs automatically in full mode, skipped in smoke mode)*

In [ ]:
if CFG['run_eval']:
    import sys, os
    sys.path.insert(0, os.path.join(REPO_DIR, 'src'))

    from evaluate import evaluate

    results = evaluate(
        checkpoint_dir = DRIVE_CKPT,
        subject        = FORGET_SUBJECT,
        n_forget       = 100,
        n_retain       = 100,
        output_dir     = f'/content/drive/MyDrive/grpo_unlearning/results',
    )

    print('\n' + '='*55)
    print(f'  Forget Score  (target > 0.70): {results["forget_score"]:.4f}')
    print(f'  Utility Score (target > 0.60): {results["utility_score"]:.4f}')
    fs_ok = results['forget_score']  > 0.70
    us_ok = results['utility_score'] > 0.60
    print(f'  Forget target met:  {"✓" if fs_ok else "✗ — may need more steps"}')
    print(f'  Utility target met: {"✓" if us_ok else "✗ — check for capability collapse"}')
    print('='*55)
else:
    print('Smoke mode — evaluation skipped.')